In [ ]:
import os
import datetime
import deltalake
import polars as pl
import pyarrow.parquet as pq
import pyarrow.fs as fs
from urllib.parse import urlparse


In [ ]:
table = deltalake.DeltaTable(
    "s3://lakehouse/silver/sales_items",
    storage_options={
        "AWS_ENDPOINT_URL": "http://[IP of MinIO]:9000",
        "AWS_REGION": "local",
        "AWS_S3_FORCE_PATH_STYLE": "true",
        "AWS_ALLOW_HTTP": "true",
    },
)


In [ ]:
arrow_table = table.to_pyarrow_table(
    filters=[("business_date", "=", datetime.date(2026, 3, 2))]
)

df = pl.from_arrow(arrow_table)
df


Number of unique rows in the Silver Sales table

In [ ]:
df.group_by("business_date").count()


In [ ]:
df.group_by("isbn").count().sort("count", descending=True)


In [ ]:
df.head(10)


In [ ]:
df.height


In [ ]:
s3 = fs.S3FileSystem(
    endpoint_override="http://[IP of MinIO]:9000",
    region="local",
)

files_31 = [f for f in table.file_uris() if "2026-03-02" in f]

counts = []
for uri in files_31:
    parsed = urlparse(uri)
    bucket = parsed.netloc
    key = parsed.path.lstrip("/")
    full_path = f"{bucket}/{key}"
    pa_tbl = pq.read_table(full_path, filesystem=s3)
    counts.append((uri, pa_tbl.num_rows))

print("\nRow counts per file:")
for f, n in counts:
    print(f"{n:>8} rows  |  {f}")

print("\nTOTAL ROWS:", sum(n for _, n in counts))


In [ ]:
table.version()


In [ ]:
hist = table.history()
pl.DataFrame(hist[:10])


In [ ]:
s3 = fs.S3FileSystem(
    endpoint_override="http://[IP of MinIO]:9000",
    region="local",
)

files = s3.get_file_info(
    fs.FileSelector("lakehouse/checkpoints/silver_sales_items/offsets")
)

offset_files = sorted(
    [f for f in files if f.type == fs.FileType.File],
    key=lambda f: int(f.base_name)
)

offset_files[-5:]


In [ ]:
avg_item_price = df["unit_price"].mean()
avg_item_price


In [ ]:
silver_items = df.filter(pl.col("business_date") == pl.date(2026, 3, 2))
silver_items.shape
